# Explore and test object filters for the Fink/LSST light curves

Uses `prepare_lsst.py` (same folder) to build the alert dataframe that `main_preprocessing_lsst.py` feeds to astro-mtan.
The goal is to **choose the thresholds of `select_objects`** before launching the full preprocessing.

1. Load and clean the alerts (`load_lsst_alerts`)
2. Compute per-object statistics once (fast, vectorized)
3. Explore cuts: grid of thresholds, histograms
4. Look at light curves kept / rejected
5. Export the selection

In [ ]:
import sys
sys.path.insert(0, ".")           # prepare_lsst.py must be in the same folder as this notebook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from prepare_lsst import load_lsst_alerts, select_objects, count_nights, BANDS

# ---- USER SETTINGS ----------------------------------------------------------------------------
DATA_PATH = "sample_300_objects.parquet"   # a ftransfer directory (full dataset) or a parquet file
FLUX_MODE = "diff"                         # 'diff' (psfFlux, SN-like) or 'science' (scienceFlux, AGN-like)
SNR_MIN = 5.0
# -----------------------------------------------------------------------------------------------

BAND_COLORS = {"g": "#3C8DFF", "r": "#15284F", "i": "#E69F00", "z": "#F5622E"}   # same palette as your notebook

## 1. Raw data and cleaned alerts

In [ ]:
raw = pd.read_parquet(DATA_PATH, dtype_backend="pyarrow")
print(f"{len(raw):,} raw rows, {raw['diaObjectId'].nunique():,} objects, "
      f"{raw['diaObjectId'].isna().sum():,} rows without diaObjectId")
print(raw["band"].value_counts().to_dict())

alerts = load_lsst_alerts(DATA_PATH, flux_mode=FLUX_MODE, snr_min=SNR_MIN)
print(f"{len(raw) - len(alerts):,} rows removed by the cleaning "
      "(no object id, duplicate, band not in griz, flux <= 0, snr < threshold)")
alerts.head()

## 2. Per-object statistics (computed once)

`select_objects` loops over objects in Python, which is slow on ~1M alerts.
For exploration we compute the per-object quantities once, then every cut is a fast boolean mask.

In [ ]:
def object_stats(alerts):
    g = alerts.groupby("objectId")
    st = pd.DataFrame({"n_points": g.size(), "t_min": g["jd"].min(), "t_max": g["jd"].max()})
    st["duration_d"] = st["t_max"] - st["t_min"]
    per_band = (alerts.groupby(["objectId", "fid"]).size().unstack(fill_value=0)
                      .reindex(columns=[1, 2, 3, 4], fill_value=0))
    per_band.columns = BANDS                                   # g, r, i, z
    st = st.join(per_band)
    night = np.floor(alerts["jd"] - 0.65)                      # same night definition as count_nights
    st["n_nights"] = night.groupby(alerts["objectId"]).nunique()
    return st


def apply_cuts(st, min_total=10, min_per_band=3, min_bands=2, min_duration_days=None, min_nights=None):
    """Boolean mask equivalent to select_objects(), on the stats table."""
    keep = (st["n_points"] >= min_total) & ((st[BANDS] >= min_per_band).sum(axis=1) >= min_bands)
    if min_duration_days is not None:
        keep &= st["duration_d"] >= min_duration_days
    if min_nights is not None:
        keep &= st["n_nights"] >= min_nights
    return keep


stats = object_stats(alerts)
stats.describe().round(2)

Sanity check: the fast mask must give exactly the same objects as `select_objects`.

In [ ]:
params = dict(min_total=10, min_per_band=3, min_bands=2, min_duration_days=None, min_nights=3)
fast = set(stats.index[apply_cuts(stats, **params)])
slow = set(select_objects(alerts, **params)["objectId"].unique())
print("identical selection:", fast == slow, f"({len(fast)} objects)")

## 3. Explore the cuts

### 3a. Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(stats["n_points"], bins=np.logspace(0, np.log10(stats["n_points"].max() + 1), 40), color="C0")
axes[0, 0].set_xscale("log"); axes[0, 0].set_xlabel("points per object")
axes[0, 1].hist(stats["duration_d"], bins=40, color="C1"); axes[0, 1].set_yscale("log")
axes[0, 1].set_xlabel("duration [days]")
axes[1, 0].hist(stats["n_nights"], bins=np.arange(0, stats["n_nights"].max() + 2), color="C2")
axes[1, 0].set_yscale("log"); axes[1, 0].set_xlabel("distinct nights")
n_bands3 = (stats[BANDS] >= 3).sum(axis=1)
axes[1, 1].bar(range(5), [(n_bands3 == k).sum() for k in range(5)], color="C3")
axes[1, 1].set_xlabel("number of bands with >= 3 points"); axes[1, 1].set_xticks(range(5))
for ax in axes.flat: ax.set_ylabel("objects"); ax.grid(alpha=0.3)
plt.tight_layout()

print("points per band (all objects):", alerts["fid"].map(dict(enumerate(BANDS, start=1))).value_counts().to_dict())

### 3b. Grid of thresholds

Number of objects kept for each combination.
Rows: minimal number of nights. Columns: (`min_per_band`, `min_bands`).
`(1, 1)` is the cut of your original notebook (any band) and `(3, 2)` is the current default.

In [ ]:
band_cuts = [(1, 1), (3, 1), (3, 2), (3, 3), (5, 2)]
night_cuts = [None, 2, 3, 5, 10]
grid = pd.DataFrame(
    {f"per_band>={mpb}, bands>={mb}": [int(apply_cuts(stats, min_per_band=mpb, min_bands=mb, min_nights=mn).sum())
                                      for mn in night_cuts]
     for mpb, mb in band_cuts},
    index=[f"nights>={mn}" if mn else "no night cut" for mn in night_cuts])
grid

### 3c. Choose your cuts

In [ ]:
CUTS = dict(min_total=10, min_per_band=3, min_bands=2, min_duration_days=None, min_nights=3)   # <-- edit

keep = apply_cuts(stats, **CUTS)
print(f"{keep.sum()} / {len(stats)} objects kept")
kept_stats = stats[keep]
print(f"points kept: {int(kept_stats['n_points'].sum()):,} / {int(stats['n_points'].sum()):,}")
print("median duration [d]:", round(kept_stats["duration_d"].median(), 1),
      "| median points:", kept_stats["n_points"].median(),
      "| median nights:", kept_stats["n_nights"].median())
print("median points per band:", kept_stats[BANDS].median().to_dict())

## 4. Look at the light curves

Magnitudes (`magpsf`, inverted axis) as they will be seen by the model, one colour per band.

In [ ]:
def plot_lc(ax, alerts, object_id, stats=None):
    sub = alerts[alerts["objectId"] == object_id].sort_values("jd")
    t0 = sub["jd"].min()
    for fid, band in enumerate(BANDS, start=1):
        s = sub[sub["fid"] == fid]
        if len(s):
            ax.errorbar(s["jd"] - t0, s["magpsf"], yerr=s["sigmapsf"], fmt="o", ms=4, lw=0.8,
                        color=BAND_COLORS[band], label=band, alpha=0.85)
    ax.invert_yaxis()
    ax.set_xlabel("days since first alert"); ax.set_ylabel("mag (AB)")
    title = str(object_id)
    if stats is not None:
        r = stats.loc[object_id]
        title += f"\nN={int(r.n_points)}, {r.duration_d:.1f} d, {int(r.n_nights)} nights"
    ax.set_title(title, fontsize=9)
    ax.legend(ncol=4, fontsize=8)


def show_sample(ids, ncols=3, seed=0):
    ids = list(ids)
    if not ids:
        print("no object"); return
    rng = np.random.default_rng(seed)
    ids = rng.choice(ids, size=min(len(ids), 2 * ncols), replace=False)
    nrows = int(np.ceil(len(ids) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.6 * nrows), squeeze=False)
    for ax, oid in zip(axes.flat, ids):
        plot_lc(ax, alerts, oid, stats)
    for ax in axes.flat[len(ids):]:
        ax.axis("off")
    plt.tight_layout()

print("KEPT objects")
show_sample(stats.index[keep])

In [ ]:
# What is rejected by your cuts but kept by the original notebook cut (> 10 alerts, any band)?
notebook_cut = apply_cuts(stats, min_per_band=1, min_bands=1)
rejected = stats.index[notebook_cut & ~keep]
print(f"{len(rejected)} objects rejected by CUTS but kept by the notebook cut")
show_sample(rejected)

## 5. Export

Saves the cleaned dataframe restricted to the selected objects, and prints the line to paste in `main_preprocessing_lsst.py`.

In [ ]:
selected = alerts[alerts["objectId"].isin(stats.index[keep])]
selected.to_parquet("selected_alerts.parquet")
print(f"saved selected_alerts.parquet: {len(selected):,} alerts, {selected['objectId'].nunique():,} objects")

args = ", ".join(f"{k}={v}" for k, v in CUTS.items())
print("\nIn main_preprocessing_lsst.py use:")
print(f"df_alerts = select_objects(df_alerts, {args})")